In [2]:
import os
import numpy as np
import pandas as pd



import calendar
from datetime import datetime

from scipy.special import gamma

np.seterr(divide='ignore',invalid='ignore')
#判断是否闰年
def isLeapYear(year):
    if not year%4 and  year%100 or not year%400:
        return 1
    return 0

def e0(T):
    T=0.6108*np.exp(17.27*T/(T+237.3))
    return T

#空值填充
def fillna(data):
    data=data.interpolate()
    data=data.T
    data=data.fillna(data.mean()) 
    data=data.T
    data=data.fillna(data.mean()) 
    return data

def d_to_jd(time):
    fmt = '%Y-%m-%d'
    dt = datetime.strptime(time, fmt)
    tt = dt.timetuple()
    return tt.tm_year * 1000 + tt.tm_yday

def jd_to_time(time):
    dt = datetime.strptime(str(time), '%Y%j').date()
    fmt ='%Y-%m-%d'
    return dt.strftime(fmt)


In [6]:
def calculate_ET0(T_avg,T1_avg,Tmin,Tmax,RH,ssh,wind_10,lat,lon,idoy,a_s,b_s,Z):   
    dr=1+0.033*np.cos(2*np.pi*(idoy+1)/365)#日地相对距离    
    lat_rad=lat*np.pi/180#纬度转弧度
    sun_rad=0.409*np.sin((idoy+1)*2*np.pi/365-1.39)#太阳角度
    Ws=np.arccos(-np.tan(lat_rad)*np.tan(sun_rad))#日落时角
    Ra=(24*60*0.082/np.pi)*dr*(Ws*np.sin(lat_rad)*np.sin(sun_rad)+np.cos(lat_rad)*np.cos(sun_rad)*np.sin(Ws))
    ssh_N=24/np.pi*Ws#最大可能日照时数
    Rs=(a_s+b_s*(ssh/ssh_N))*Ra
    Rns=0.77*Rs
    Rs0=(0.75+0.00002*Z)*Ra
    es=(e0(Tmax)+e0(Tmin))/2
    ea=(RH/100)*es
    Rnl=4.903*10**-9*(((Tmax+273.16)**4+(Tmin+273.16)**4)/2)*(0.34-0.14*ea**0.5)*(-0.35+1.35*(Rs/Rs0))      
    Rn=Rns-Rnl
    
    delt=4098*(0.6108*np.exp(17.27*T_avg/(T_avg+237.3)))/(T_avg+237.3)**2    
    G=0.14*(T_avg-T1_avg)
    a=0.408*delt*(Rn-G)
    
    #计算b
    if is_win_to_2m==1:
        v2=wind_10*4.87/np.log(67.8*10-5.42)
    else:
        v2=wind_10
    b=(60.62805/(T_avg+273))*v2*(es-ea)*((293-0.0065*Z)/293)**5.26
    #计算d
    d=0.0673645*((293-0.0065*Z)/293)**5.26*(1+0.34*v2)     
    
    #计算ET0
    ET0i=(a+b)/(delt+d)
    return ET0i


#计算SPEI  

def Calculat_SPEI(Time_data):
    result_list=[]
    c0,c1,c2,d1,d2,d3=2.515517,0.802853,0.010328,1.432788,0.189269,0.001308
    Time_data_sort=sorted(Time_data)
    Fi=[(x-0.35)/len(Time_data_sort) for x in range(1,len(Time_data_sort)+1)]
    Fi1=[(1-Fi[ind])*v for ind,v in enumerate(Time_data_sort)]
    Fi2=[(1-Fi[ind])**2*v for ind,v in enumerate(Time_data_sort)]
    w0=np.mean(Time_data_sort)
    w1=np.mean(Fi1)
    w2=np.mean(Fi2)
    b_=(2*w1-w0)/(6*w1-w0-6*w2)
    a_=(w0-2*w1)*b_/(gamma(1+1/b_)*gamma(1-1/b_))

    r=w0-a_*gamma(1+1/b_)*gamma(1-1/b_)
    Fx=[(1+abs(a_/(v-r))**b_)**-1  for v in Time_data ]
    
    P=[]
    for v in Fx:
        if v<=0.5:
            P.append(v)
        else:
            P.append(1-v)
    w=[(-2*np.log(v))**0.5 for v in P]
    for ind,v in enumerate(Fx):
        if v<=0.5:
            SPEI=-(w[ind]-(c0+c1*w[ind]+c2*w[ind]**2)/(1+d1*w[ind]+d2*w[ind]**2+d3*w[ind]**3))
        else:
            SPEI=w[ind]-(c0+c1*w[ind]+c2*w[ind]**2)/(1+d1*w[ind]+d2*w[ind]**2+d3*w[ind]**3)
        result_list.append(SPEI)   
    return result_list

#台站数据整理
#风速 平均气温 日降水量 日照数 相对湿度 最低气温 最高气温
DataNameDict={'R20':'日降水量','Tmax':'最高气温','Tmin':'最低气温','T':'平均气温',
            'S':'日照数','U':'相对湿度','WS':'风速'}

workpath=r'E:\SPEI计算'
datapath=os.path.join(workpath,'daily_climate缺测插补')

ET0_path=os.path.join(workpath,'EI0')
SPEI_path=os.path.join(workpath,'SPEI')

SPEI_daily_path=os.path.join(SPEI_path,'daily')
SPEI_month_path=os.path.join(SPEI_path,'Month')



is_win_to_2m=1#风速是否要转为2m风速,1是0否

startyear,endyear=1969,2022
stinfo=pd.read_excel(os.path.join(datapath,'站点编号及信息_选定_1969-2022.xlsx'))

stinfo['station'] = stinfo['station'].apply(str)
station_list=list(stinfo['station'])
Z_list=list(stinfo['Z'])
a_s_list=list(stinfo['a_s'])
b_s_list=list(stinfo['b_s'])
lat_list=list(stinfo['lat'])
lon_list=list(stinfo['lon'])  
st_sum=len(station_list)

In [8]:
#step 01 计算逐日ET0 和降水蒸散差 D0=PER-EI0

fnpath=datapath
outpath=ET0_path

dt0=datetime.strptime(str(startyear-1)+'1231','%Y%m%d')
print('计算ET0、Di......')
for i in range(st_sum): 
    st_no=stinfo['station'][i]
    #获取站点参数
    Z=Z_list[i]
    a_s=a_s_list[i]
    b_s=b_s_list[i] 
    lat=lat_list[i]
    lon=lon_list[i]
    #气象数据读取        
    st_no=stinfo['station'][i]
    fn=os.path.join(fnpath,str(st_no)+".csv")
    dfs=pd.read_csv(fn)
    dfs=dfs.dropna(axis=0,how='any')
    dfs['doys']=dfs['time'].map(d_to_jd) 
    dfs['time']=pd.to_datetime(dfs['time'])
    dfs['doys']=dfs['doys']-dfs['time'].dt.year*1000
    dfs.sort_values('time')
    dfs=dfs[dfs['time']>=dt0]
    data_Tavg=list(dfs['T'])
    data_Tmax=list(dfs['Tmax'])
    data_Tmin=list(dfs['Tmin'])
    data_Tpre=list(dfs['R20'])
    data_Tsun=list(dfs['S'])
    data_Thum=list(dfs['U'])
    data_Twin=list(dfs['WS'])
    data_Time=list(dfs['time'])
    data_doys=list(dfs['doys'])
    data_ET0=[]
    data_Di=[]

    for k in range(1,len(data_Time)):    
        T=data_Tavg[k]#平均气温
        T1=data_Tavg[k-1]
        if np.isnan(T1): T1=T
        Tmax=data_Tmax[k]
        Tmin=data_Tmin[k]
        Per=data_Tpre[k]
        n=data_Tsun[k]#日照时数
        hum=data_Thum[k]#相对湿度
        ws=data_Twin[k]#风速   
        idoy=data_doys[k]#日序
        #计算逐日蒸散
        ET0=calculate_ET0(T,T1,Tmin,Tmax,hum,n,ws,lat,lon,idoy,a_s,b_s,Z)
        data_ET0.append(ET0)            
        #计算Di
        Di=Per-ET0
        data_Di.append(Di)
    outdf=pd.DataFrame()
    outdf['time']=data_Time[1:]
    outdf['ET0']=data_ET0
    outdf['DI']=data_Di
    outfn=os.path.join(outpath,str(st_no)+".csv")
    outdf.to_csv(outfn)
    print(i,'/',st_sum,st_no,outfn)
print(datetime.now())

计算ET0、Di......
0 / 115 56444 E:\SPEI计算\EI0\56444.csv
1 / 115 56483 E:\SPEI计算\EI0\56483.csv
2 / 115 56489 E:\SPEI计算\EI0\56489.csv
3 / 115 56497 E:\SPEI计算\EI0\56497.csv
4 / 115 56533 E:\SPEI计算\EI0\56533.csv
5 / 115 56543 E:\SPEI计算\EI0\56543.csv
6 / 115 56548 E:\SPEI计算\EI0\56548.csv
7 / 115 56567 E:\SPEI计算\EI0\56567.csv
8 / 115 56582 E:\SPEI计算\EI0\56582.csv
9 / 115 56585 E:\SPEI计算\EI0\56585.csv
10 / 115 56586 E:\SPEI计算\EI0\56586.csv
11 / 115 56594 E:\SPEI计算\EI0\56594.csv
12 / 115 56595 E:\SPEI计算\EI0\56595.csv
13 / 115 56596 E:\SPEI计算\EI0\56596.csv
14 / 115 56651 E:\SPEI计算\EI0\56651.csv
15 / 115 56652 E:\SPEI计算\EI0\56652.csv
16 / 115 56654 E:\SPEI计算\EI0\56654.csv
17 / 115 56664 E:\SPEI计算\EI0\56664.csv
18 / 115 56669 E:\SPEI计算\EI0\56669.csv
19 / 115 56684 E:\SPEI计算\EI0\56684.csv
20 / 115 56688 E:\SPEI计算\EI0\56688.csv
21 / 115 56697 E:\SPEI计算\EI0\56697.csv
22 / 115 56739 E:\SPEI计算\EI0\56739.csv
23 / 115 56742 E:\SPEI计算\EI0\56742.csv
24 / 115 56745 E:\SPEI计算\EI0\56745.csv
25 / 115 56746 E:\SP

In [ ]:
##step 02 计算逐日SPEI
fnpath=ET0_path
outpath=SPEI_daily_path
dt0=datetime.strptime(str(startyear+1)+'0101','%Y%m%d')
print("step 02 计算逐日SPEI.....")
for i in range(0,st_sum): 
    st_no=stinfo['station'][i]
    fn=os.path.join(fnpath,str(st_no)+".csv")
    df=pd.read_csv(fn)
    df['doys']=df['time'].map(d_to_jd) 
    df['time']=pd.to_datetime(df['time'])
    
    df['doys']=df['doys']-df['time'].dt.year*1000
    df['DI_30']=df['DI'].rolling(30).sum()
    df.sort_values('time')
    df=df[df['time']>=dt0]
    dfs=pd.DataFrame()
    for k in range(1,367):
        outdf=pd.DataFrame()
        data_SPEI=[]
        data_Time=list(df[df['doys']==k]['time'])   
        data_DI=list(df[df['doys']==k]['DI_30'])            
        data_SPEI=Calculat_SPEI(data_DI)
        outdf['time']=data_Time
        outdf['SPEI']=data_SPEI
        dfs=pd.concat([dfs,outdf])
    dfs['time']=pd.to_datetime(dfs['time'])
    dfs=dfs.sort_values('time')
    outfn=os.path.join(outpath,str(st_no)+".csv")
    print(i,st_sum,st_no,outfn)
    dfs.to_csv(outfn)
print('This is OK',datetime.now())

step 02 计算逐日SPEI.....
0 115 56444 E:\SPEI计算\SPEI\daily\56444.csv
1 115 56483 E:\SPEI计算\SPEI\daily\56483.csv
2 115 56489 E:\SPEI计算\SPEI\daily\56489.csv
3 115 56497 E:\SPEI计算\SPEI\daily\56497.csv
4 115 56533 E:\SPEI计算\SPEI\daily\56533.csv
5 115 56543 E:\SPEI计算\SPEI\daily\56543.csv
6 115 56548 E:\SPEI计算\SPEI\daily\56548.csv
7 115 56567 E:\SPEI计算\SPEI\daily\56567.csv
8 115 56582 E:\SPEI计算\SPEI\daily\56582.csv
9 115 56585 E:\SPEI计算\SPEI\daily\56585.csv
10 115 56586 E:\SPEI计算\SPEI\daily\56586.csv
11 115 56594 E:\SPEI计算\SPEI\daily\56594.csv
12 115 56595 E:\SPEI计算\SPEI\daily\56595.csv
13 115 56596 E:\SPEI计算\SPEI\daily\56596.csv
14 115 56651 E:\SPEI计算\SPEI\daily\56651.csv
15 115 56652 E:\SPEI计算\SPEI\daily\56652.csv
16 115 56654 E:\SPEI计算\SPEI\daily\56654.csv
17 115 56664 E:\SPEI计算\SPEI\daily\56664.csv
18 115 56669 E:\SPEI计算\SPEI\daily\56669.csv


In [13]:
##step 03 计算月时间尺度SPEI

fnpath=ET0_filepath
outpath=SPEI_month_path
startyear=1969
fd_names=['SPEI1_Month','SPEI2_Month','SPEI3_Month','SPEI4_Month','SPEI5_Month','SPEI6_Month'
          ,'SPEI7_Month','SPEI8_Month','SPEI9_Month','SPEI10_Month','SPEI11_Month','SPEI12_Month']
dt0=datetime.strptime(str(startyear+1)+'0101','%Y%m%d')
print('计算不同时间尺度的SPEI')
for i in range(st_sum):   
    st_no=stinfo['station'][i]
    fn=os.path.join(fnpath,str(st_no)+".csv")
    df=pd.read_csv(fn)   
    df['time']=pd.to_datetime(df['time'])
    df['year']=df['time'].dt.year
    df['month']=df['time'].dt.month    
    month_df=df.groupby(['year','month'])['DI'].sum().reset_index()
    month_df['DI2_Month']=month_df['DI'].rolling(2).sum()
    month_df['DI3_Month']=month_df['DI'].rolling(3).sum()
    month_df['DI4_Month']=month_df['DI'].rolling(4).sum()
    month_df['DI5_Month']=month_df['DI'].rolling(5).sum()
    month_df['DI6_Month']=month_df['DI'].rolling(6).sum()
    month_df['DI7_Month']=month_df['DI'].rolling(7).sum()
    month_df['DI8_Month']=month_df['DI'].rolling(8).sum()
    month_df['DI9_Month']=month_df['DI'].rolling(9).sum()
    month_df['DI10_Month']=month_df['DI'].rolling(10).sum()
    month_df['DI11_Month']=month_df['DI'].rolling(11).sum()
    month_df['DI12_Month']=month_df['DI'].rolling(12).sum()
    
    month_df=month_df[(month_df['year']>startyear)&(month_df['year']<=endyear)]
    dfs=pd.DataFrame()
    for im in range(1,13):
        outdf=pd.DataFrame()
        data_SPEI=[]
        data_year=list(month_df[month_df['month']==im]['year'])           
        data_1=list(month_df[month_df['month']==im]['DI']) 
        data_2=list(month_df[month_df['month']==im]['DI2_Month']) 
        data_3=list(month_df[month_df['month']==im]['DI3_Month']) 
        data_4=list(month_df[month_df['month']==im]['DI4_Month'])   
        data_5=list(month_df[month_df['month']==im]['DI5_Month'])         
        data_6=list(month_df[month_df['month']==im]['DI6_Month']) 
        data_7=list(month_df[month_df['month']==im]['DI7_Month']) 
        data_8=list(month_df[month_df['month']==im]['DI8_Month']) 
        data_9=list(month_df[month_df['month']==im]['DI9_Month'])   
        data_10=list(month_df[month_df['month']==im]['DI10_Month'])         
        data_11=list(month_df[month_df['month']==im]['DI11_Month']) 
        data_12=list(month_df[month_df['month']==im]['DI12_Month']) 
        
        outdf['year']=data_year
        outdf['month']=im
        k=0
        for list0 in [data_1,data_2,data_3,data_4,data_5,data_6,data_7,data_8,data_9,data_10,data_11,data_12]:
            data_SPEI=Calculat_SPEI(list0)              
            fd=fd_names[k]
            outdf[fd]=data_SPEI
            k=k+1
        dfs=pd.concat([dfs,outdf])
    

    outfn=os.path.join(outpath,str(st_no)+".csv")
    print(i,st_sum,st_no,outfn)
    dfs.to_csv(outfn)
print(outfn,datetime.now())       


计算不同时间尺度的SPEI
0 115 56444 E:\data\SPEI\month\56444.csv
1 115 56483 E:\data\SPEI\month\56483.csv
2 115 56489 E:\data\SPEI\month\56489.csv
3 115 56497 E:\data\SPEI\month\56497.csv
4 115 56533 E:\data\SPEI\month\56533.csv
5 115 56543 E:\data\SPEI\month\56543.csv
6 115 56548 E:\data\SPEI\month\56548.csv
7 115 56567 E:\data\SPEI\month\56567.csv
8 115 56582 E:\data\SPEI\month\56582.csv
9 115 56585 E:\data\SPEI\month\56585.csv
10 115 56586 E:\data\SPEI\month\56586.csv
11 115 56594 E:\data\SPEI\month\56594.csv
12 115 56595 E:\data\SPEI\month\56595.csv
13 115 56596 E:\data\SPEI\month\56596.csv
14 115 56651 E:\data\SPEI\month\56651.csv
15 115 56652 E:\data\SPEI\month\56652.csv
16 115 56654 E:\data\SPEI\month\56654.csv
17 115 56664 E:\data\SPEI\month\56664.csv
18 115 56669 E:\data\SPEI\month\56669.csv
19 115 56684 E:\data\SPEI\month\56684.csv
20 115 56688 E:\data\SPEI\month\56688.csv
21 115 56697 E:\data\SPEI\month\56697.csv
22 115 56739 E:\data\SPEI\month\56739.csv
23 115 56742 E:\data\SPEI\mont